# Qwen3-VL-8B Hebrew OCR fine-tune (Unsloth, Colab A100)

Trains vision **and** language layers with QLoRA on the Talmud dataset built by
`src/finetuning/qwen_hebrew/build_dataset.py` (pushed to a private HF repo with `--push_to_hub`).

**Run the guardrail cells (4 and 5) before any real training.** They exist because a
previous fine-tuning attempt in this project trained for days with images never
reaching the model and loss computed on prompt tokens — this notebook makes those
failures impossible to miss.

Runtime: A100 (40GB). T4 will OOM at these sequence lengths.

In [ ]:
# Cell 1 — installs (pin unsloth; let it pick matching torch/transformers)
%pip install -q "unsloth[colab-new]" "datasets>=3.0" wandb

import torch
assert torch.cuda.is_available(), "No GPU — switch the runtime to A100"
print(torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 — load the dataset from the private hub repo
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

login(token=userdata.get("HF_TOKEN"))  # add HF_TOKEN in Colab secrets

DATASET_REPO = "CHANGE_ME/qwen-hebrew-talmud"  # from build_dataset.py --push_to_hub

train_all = load_dataset(DATASET_REPO, split="train")
smoke = load_dataset(DATASET_REPO, split="smoke")
val = load_dataset(DATASET_REPO, split="val")

crops = train_all.filter(lambda t: t == "crop_transcribe", input_columns="task")
pages = train_all.filter(lambda t: t == "page_extract", input_columns="task")
print(f"crops={len(crops)}  pages={len(pages)}  val={len(val)}  smoke={len(smoke)}")

In [ ]:
# Cell 3 — model: Qwen3-VL-8B 4bit, LoRA on vision + language (joint training)
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,      # user decision: attack the vision encoder directly
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    random_state=3407,
)

# Resolution policy: never let the processor shrink dense Hebrew below readability.
ip = tokenizer.image_processor if hasattr(tokenizer, "image_processor") else None
if ip is not None and hasattr(ip, "min_pixels"):
    ip.min_pixels = 256 * 28 * 28
    ip.max_pixels = 4_500_000  # A100: full pages at native resolution
    print(f"resolution policy: min={ip.min_pixels} max={ip.max_pixels}")

In [ ]:
# Cell 4 — format mapping: dataset columns -> Unsloth conversation format
def to_conversation(sample):
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text", "text": sample["question"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["answer"]},
            ]},
        ]
    }

converted_smoke = [to_conversation(smoke[i]) for i in range(len(smoke))]
print(converted_smoke[0]["messages"][0]["content"][1]["text"][:100])

In [ ]:
# Cell 5 — MANDATORY collator guardrail. Do not train if this cell fails.
#
# Guards against the two root causes of the failed Gemma fine-tune:
#   (a) images silently not reaching the model (pixel_values was None)
#   (b) loss computed on prompt/padding tokens (labels never masked)
from unsloth.trainer import UnslothVisionDataCollator

collator = UnslothVisionDataCollator(model, tokenizer)
batch = collator([converted_smoke[0], converted_smoke[1]])

# (a) images must flow
pv = batch.get("pixel_values")
assert pv is not None, "pixel_values is None — images are NOT reaching the model"
assert float(pv.abs().sum()) > 0, "pixel_values all-zero — image preprocessing broken"
print(f"pixel_values OK: shape={tuple(pv.shape)}")

# (b) labels must be masked down to the assistant answer
labels = batch["labels"]
unmasked = labels[0][labels[0] != -100]
decoded = tokenizer.tokenizer.decode(unmasked) if hasattr(tokenizer, "tokenizer") else tokenizer.decode(unmasked)
answer = converted_smoke[0]["messages"][1]["content"][0]["text"]
assert answer[:40] in decoded, (
    f"Unmasked labels don't contain the assistant answer.\nDecoded: {decoded[:200]}"
)
assert converted_smoke[0]["messages"][0]["content"][1]["text"][:40] not in decoded, (
    "Prompt text found in unmasked labels — loss is leaking onto the prompt"
)
frac = float((labels[0] != -100).float().mean())
print(f"label masking OK: {frac:.1%} of tokens in the loss (answer only)")

In [ ]:
# Cell 6 — overfit-8 sanity: pipeline must drive loss ~0 on 8 fixed samples
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported

FastVisionModel.for_training(model)

overfit_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=converted_smoke[:8],
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        max_steps=300,
        learning_rate=1e-4,
        logging_steps=25,
        optim="adamw_8bit",
        lr_scheduler_type="constant",
        seed=3407,
        output_dir="outputs_overfit",
        report_to="none",
        bf16=is_bf16_supported(),
        fp16=not is_bf16_supported(),
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=8192,
    ),
)
stats = overfit_trainer.train()
final_loss = stats.training_loss
print(f"final mean loss {final_loss:.4f}")
assert final_loss < 0.15, (
    f"Overfit-8 failed (loss {final_loss:.3f} >= 0.15) — the pipeline cannot even\n"
    "memorize 8 samples. Something is broken; do NOT run full training."
)

In [ ]:
# Cell 7 — eyeball an overfit generation vs its ground truth
FastVisionModel.for_inference(model)
sample = smoke[0]
messages = [{"role": "user", "content": [
    {"type": "image"}, {"type": "text", "text": sample["question"]},
]}]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(sample["image"], input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=512, temperature=0.0, do_sample=False)
print("GENERATED:", tokenizer.batch_decode(out)[0][-600:])
print("\nGROUND TRUTH:", sample["answer"][:400])

In [ ]:
# Cell 8 — full run: 85% crops / 15% pages, 2 epochs, W&B
import wandb
from datasets import interleave_datasets

wandb.login(key=userdata.get("WANDB_API_KEY"))

mixture = interleave_datasets(
    [crops, pages], probabilities=[0.85, 0.15], seed=3407,
    stopping_strategy="all_exhausted",
)
mixture = mixture.map(to_conversation)

FastVisionModel.for_training(model)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=mixture,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=1e-4,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        logging_steps=10,
        save_steps=500,
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=3407,
        output_dir="outputs_full",
        report_to="wandb",
        run_name="colab_joint_qlora_crops85_pages15",
        bf16=is_bf16_supported(),
        fp16=not is_bf16_supported(),
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=8192,
    ),
)
trainer.train()

In [ ]:
# Cell 9 — export merged fp16 weights and push to a private hub repo
MERGED_REPO = "CHANGE_ME/qwen3-vl-8b-hebrew-merged"
model.save_pretrained_merged("qwen3-vl-8b-hebrew-merged", tokenizer, save_method="merged_16bit")
model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", private=True)
print(f"pushed {MERGED_REPO}")

## Back on the Mac: convert to MLX and load in LM Studio

```bash
# 8-bit MLX conversion of the merged model
.venv-mlx/bin/python -m mlx_vlm.convert \
    --hf-path CHANGE_ME/qwen3-vl-8b-hebrew-merged \
    --mlx-path models/qwen3-vl-8b-heb-colab -q --q-bits 8

# import into LM Studio, then benchmark before/after:
lms import models/qwen3-vl-8b-heb-colab
python -m src.datasets.evaluations.talmud_evaluation \
    --lm_studio_models qwen/qwen3-vl-8b,<lm-studio-id-of-fine-tune>
```

Also run the guardrail probe on the converted model before benchmarking:
```bash
.venv-mlx/bin/python -m src.finetuning.qwen_hebrew.image_dependence_probe \
    --model models/qwen3-vl-8b-heb-colab
```